In [35]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras

from sklearn.model_selection import train_test_split
from keras.models import Model
from keras.optimizers import AdamW
from keras.losses import MeanSquaredError

from keras.layers import Input, Embedding, Dot, Flatten

In [2]:
df = pd.read_csv("/mnt/e/Deep Learning/data/ratings.csv")
df

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4
...,...,...,...
981751,10000,48386,5
981752,10000,49007,4
981753,10000,49383,5
981754,10000,50124,5


In [3]:
df.info

<bound method DataFrame.info of         book_id  user_id  rating
0             1      314       5
1             1      439       3
2             1      588       5
3             1     1169       4
4             1     1185       4
...         ...      ...     ...
981751    10000    48386       5
981752    10000    49007       4
981753    10000    49383       5
981754    10000    50124       5
981755    10000    51328       1

[981756 rows x 3 columns]>

In [4]:
df['book_id'].value_counts()

book_id
1       100
2       100
3       100
4       100
5       100
       ... 
9315     36
1935     34
9486     24
9345     11
7803      8
Name: count, Length: 10000, dtype: int64

In [5]:
len(sorted(list(df['book_id'].value_counts().keys())))

10000

In [6]:
df['user_id'].value_counts()

user_id
12874    200
30944    200
28158    199
52036    199
12381    199
        ... 
29736      2
18936      2
43623      2
24406      2
27590      2
Name: count, Length: 53424, dtype: int64

In [7]:
len(sorted(list(df['user_id'].value_counts().keys())))

53424

In [8]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [9]:
print(train_df.shape)

(785404, 3)


In [10]:
print(test_df.shape)

(196352, 3)


In [11]:
train_df['book_id'].unique()

array([3423, 9811, 6485, ..., 8971, 1125, 9486], shape=(10000,))

In [12]:
train_df['user_id'].unique()

array([ 4608, 36373,  2957, ...,  2422, 35268, 46181], shape=(53057,))

In [13]:
test_df['user_id'].unique()

array([19643,  8563, 52110, ..., 45937,  8079, 24285], shape=(40306,))

In [14]:
train_df

,book_id,user_id,rating
341848,3423,4608,2
964349,9811,36373,5
645459,6485,2957,4
74960,750,42400,3
358670,3591,36886,5
...,...,...,...
259178,2594,26266,4
365838,3663,27212,5
131932,1320,31839,4
671155,6746,34952,2


In [18]:
train_data_book = train_df['book_id'].values
train_data_book

array([3423, 9811, 6485, ..., 1320, 6746, 1220], shape=(785404,))

In [20]:
train_data_user = train_df['user_id'].values
train_data_user

array([ 4608, 36373,  2957, ..., 31839, 34952, 32923], shape=(785404,))

In [21]:
train_labels = train_df['rating'].values
train_labels

array([2, 5, 4, ..., 4, 2, 3], shape=(785404,))

In [ ]:
train_loader_book = tf.data.Dataset.from_tensor_slices(train_data_book).batch(64)
train_loader_user = tf.data.Dataset.from_tensor_slices(train_data_user).batch(64)
train_loader_label = tf.data.Dataset.from_tensor_slices(train_labels).batch(64)

print(tf.data.experimental.cardinality(train_loader_book))
print(tf.data.experimental.cardinality(train_loader_user))
print(tf.data.experimental.cardinality(train_loader_label))

tf.Tensor(12272, shape=(), dtype=int64)
tf.Tensor(12272, shape=(), dtype=int64)
tf.Tensor(12272, shape=(), dtype=int64)


In [47]:
val_size = int(0.2 * len(train_data_user))

In [48]:
raw_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
            "Input-user": train_data_user,
            "Input-book": train_data_book
        },
        train_labels
    )
)

In [49]:
raw_ds = raw_ds.shuffle(10000)

In [50]:
train_ds = raw_ds.skip(val_size)
val_ds = raw_ds.take(val_size)

In [51]:
train_ds = train_ds.batch(256).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(256).prefetch(tf.data.AUTOTUNE)

In [33]:
user_num = df['user_id'].unique().shape[0]
book_num = df['book_id'].unique().shape[0]

print(f"Number  of users: {user_num}")
print(f"Number of books: {book_num}")

Number  of users: 53424
Number of books: 10000


In [36]:
input_book = Input(shape=(1,), name="Input-book")
embedding_book = Embedding(
    input_dim=book_num + 1,
    output_dim=5,
    name="Book-Embedding"
)(input_book)

book_vec = Flatten()(embedding_book)

input_user = Input(shape=(1,), name="Input-user")
embedding_user = Embedding(
    input_dim=user_num + 1,
    output_dim=5,
    name="User-Embedding"
)(input_user)

user_vec = Flatten()(embedding_user)

output = Dot(axes=1)([book_vec, user_vec])


model = Model(
    inputs=[input_book, input_user],
    outputs=output
)

In [38]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input-book          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input-user          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Book-Embedding      │ (None, 1, 5)      │     50,005 │ Input-book[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ User-Embedding      │ (None, 1, 5)      │    267,125 │ Input-user[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 5)         │          0 │ Book-Embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 5)         │          0 │ User-Embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 317,130 (1.21 MB)

 Trainable params: 317,130 (1.21 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# !pip install pydot

In [56]:
keras.utils.plot_model(model, show_shapes=True)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [40]:
model.compile(optimizer=AdamW(learning_rate=0.001), 
              loss=MeanSquaredError,)

In [53]:
history = model.fit(x=train_ds, validation_data=val_ds, epochs=20)

Epoch 1/20
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 15.7039 - val_loss: 14.9151
Epoch 2/20
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 12.2172 - val_loss: 9.2699
Epoch 3/20
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 6.6566 - val_loss: 4.9270
Epoch 4/20
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 3.4929 - val_loss: 2.9458
Epoch 5/20
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 2.1139 - val_loss: 2.0818
Epoch 6/20
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 1.4945 - val_loss: 1.6662
Epoch 7/20
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 1.1798 - val_loss: 1.4369
Epoch 8/20
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 1.0004 - val_loss: 1.2952
Epoch 9/20
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.8881 - val_loss: 1.2025
Epoch 10/20
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.8125 - val_loss: 1.1326
Epoch 11/20
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.7600 - val_loss: 1.0837
Epoch 12/20
2455/2455 